## What is Spark ?

### Let's keep it simple. 

The intuitive way: Spark distributes tasks to different Worker nodes (machine executing the tasks).  
The Kubernetes way: Spark distributes tasks to different containers. The location of containers among the different worker nodes is then handled by Kubernetes. The more computation power you need, the more containers get created. It's a smooth way to save Worker nodes resources.
With Kubernetes, we just speak in terms of Pods rathen than containers.

Pretty handy right ? If you want to have a deeper look on how it's done, you can by clicking on the link bellow but you still have some work to do so don't waste your time ! You can keep it to the end.

### If you want more !

#### Spark Context

The **Spark Context** is an object that hides the complexity of the underlying infrastructure to perform computation from the Data Scientist.

This Spark context is a JVM process that gives access to a Spark **driver** which schedules the tasks and spans tasks across Worker nodes through Executors.
In brief, the Spark driver communicates with all the Worker nodes. 

Each Worker node consists of one or more Executor(s) who are responsible for running the Task. Executors register themselves with Driver. The Driver has all the information about the Executors at all the time.

This working combination of Driver and Workers is known as **Spark Application**.

JVM: Java virtual machine that load, verifies and executes Java bytecode.

![](img/spark_context.png)

The Spark Application is launched with the help of the **Cluster Manager**. Spark is dependent on the **Cluster Manager** to launch the Executors and also the Driver (in Cluster mode). 

Spark can be run with any of the 5 following **Cluster Manager** :

- local : Driver use CPU threads on your local machine
- Spark Standalone Mode : A basic resource manager provided by Spark
- YARN : the historical resource manager commonly used in traditional Big Data infrastructures (Courdera or Hortonworks cluster for instance)
- Mesos : the resource manager from Berkeley University
- Kubernetes : the game changing containers orchestrator 

#### Spark on a Kubernetes cluster

![](img/k8spark_cluster_mode.png)

The Spark driver runs on a Kubernetes Pod and creates executors running within Kubernetes pods. In simple terms, Spark driver is in a Pod and it distributes tasks to different Pods. The location of Pods among the different worker nodes is then handled by Kubernetes. 

The Kubernetes **Scheduler** checks if pods are assigned to nodes. For every Pod that the scheduler discovers, the scheduler is responsible of finding the best node for that Pod to run on.  
The Kubernetes **API Server** allows to interact with Pods. It schedules executor Pods by creating or deleting Pods.


## Create a Spark session


The minimal configuration :
``` python
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
   .appName("MyApp") \
   .master("local[*]") \
   .getOrCreate()

```

local[*] will use all the cpu available on the machine


In [ ]:
# See the default configuration on the datalab
! ls $SPARK_HOME
! cat /usr/local/lib/spark/conf/spark-defaults.conf

In [35]:
# Create a more complex configuration :

spark = SparkSession \
         .builder \
         .config("spark.executor.memory", "3g") \
         .config("spark.dynamicAllocation.enabled","true") \
         .config("spark.dynamicAllocation.initialExecutors","1") \
         .config("spark.dynamicAllocation.minExecutors","1") \
         .config("spark.dynamicAllocation.maxExecutors","5") \
         .getOrCreate()

# You have regrets ? Then modify your config !

spark.conf.set("spark.sql.shuffle.partitions", "150")


In [ ]:
spark

⚠️
At the end, please stop your session to release the ressources ! Always stop it
⚠️

```python
spark.stop()
```

In [ ]:
# See the current number of executors (one for now)
! kubectl get pods -l spark-role=executor


# Configuraion
spark._jsc.hadoopConfiguration().set("fs.s3.useRequesterPaysHeader","true")
```

In [38]:
spark.stop()

### New since spark 4 
- Spark connect  

Spark Connect is a new client-server architecture. It decouples the user application from the Spark cluster .Spark Connect separates the Spark client from the Spark driver, allowing you to write Spark applications that run remotely through a thin client rather than requiring the full Spark runtime locally.

 ```
Architecture
┌─────────────┐          gRPC            ┌──────────────┐
│   Client    │ ◄──────────────────────► │ Spark Server │
│ (Thin API)  │   (Network Protocol)     │  (Driver)    │
└─────────────┘                          └──────────────┘
                                                  │
                                                  ▼
                                          ┌──────────────┐
                                          │   Executors  │
                                          └──────────────┘

```
The way Spark traditionally works is that everything is consolidated on a monolithic driver. This driver handles the distributed execution engine, scheduler, optimizer, and all the business logic packaged together. The challenge is that using Spark from environments not connected to the JVM directly is hard.

Spark Connect addresses this by bringing the full power of Spark via a client API to allow various application types to connect seamlessly. The majority of users interact with Spark using the DataFrame API and SQL. Spark Connect translates these structured APIs into “unresolved plans” that are sent to the server.

The server then processes these plans through analysis, optimization, scheduling, and execution. Results are streamed back to the client using gRPC and Apache Arrow. This use of Arrow is very convenient for connecting to other systems that speak Arrow, like Polars or data structures directly improving interoperability.

Key Benefits
1. Lightweight client: The Python client (pyspark-client) is only ~1.5 MB vs. the full PySpark installation

2. Stability: Client failures don't crash the Spark application running on the server

3. Multiple languages: Supports Python, Scala, Java, Go, and Swift clients

4. Upgradeability: Upgrade client independently from server

5. Remote execution: Run Spark from notebooks, IDEs, or applications without local Spark installation

How to Use It

```python
from pyspark.sql import SparkSession

# Connect to remote Spark server
spark = SparkSession.builder \
    .remote("sc://hostname:15002") \
    .getOrCreate()
```

